# proto-REINFORCE, notebook 1: Old Bessie

On page 3 of Old Bessie, proto-REINFORCE learned to manage the van: a table of bets, its probability table,
1,000 runs and their average $G_1$, then a small change of one bet at a time. In this notebook you build it in Python,
one small piece at a time: a roulette that picks the van's actions, runs of the van, their average, the slope of a
bet, and at the end the whole loop.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
game is in the kit: your code never deals with dice.

Start with the two folded cells below: Old Bessie's kit and the checks.

In [ ]:
#@title Old Bessie's kit (run me first) { display-mode: "form" }
# Old Bessie's kit. Run this cell first; you do not need to read it.
import random                   # Python's random numbers: random.randint(0, 3) gives 0, 1, 2 or 3
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

STATES = ['HEALTHY', 'WORN', 'SHAKY', 'FAILING']
ACTIONS = ['RUN', 'SERVICE', 'REPLACE']
HEALTHY, WORN, SHAKY, FAILING = 0, 1, 2, 3       # the rows of a table: the van's states
RUN, SERVICE, REPLACE = 0, 1, 2                  # the columns of a table: the actions
GAMMA = 0.9                # a week later, a dollar counts 0.9 times as much
WEEKS = 13                 # the weeks of a quarter: one run
RUNS = 1000                # the runs of every average, as on page 3

# The van of the arcade, as on page 1 (bessie.js): the week's money and the chances of what happens
_KIT_PROFIT = [95, 72, 40, 16]          # RUN: the route's money, by state
_KIT_BREAKDOWN = [0.02, 0.08, 0.28, 0.55]    # RUN: the chance of a breakdown, by state
_KIT_TOW = 280                       # a breakdown costs this on top, and the van is FAILING next week
_KIT_WEAR = [[0.35, 0.55, 0.1], [0.3, 0.55, 0.15], [0.45, 0.55, 0.0], [0.65, 0.35, 0.0]]     # RUN without a breakdown: wears 0, 1 or 2 levels
_KIT_SHOP = 50                       # SERVICE: a week in the garage
_KIT_REPAIR = [[1.0, 0.0, 0.0], [0.05, 0.7, 0.25], [0.45, 0.5, 0.05], [0.55, 0.42, 0.03]]     # SERVICE: repairs 0, 1 or 2 levels
_KIT_NEW_VAN = 130                   # REPLACE: a new van, HEALTHY next week
_KIT_START_BETS = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]   # page 3's bets at random
_KIT_BEST_LINE = 244.03          # the best strategy's average G1 on the luck of the loop's last update
_KIT_FIRST_LUCK = 12                 # the luck of page 3's start: B.dice(12, 1000)
_KIT_LUCK_BASE = 2000               # update i of the loop plays the luck of seed 2000 + i, as page 3's movie
_KIT_PICKS = 2                       # the seed of Python's random for the loop's picks (play sets it)


def _kit_outcomes(state, action):
    """Every outcome of a week, in the order of the page: [running chance, next state, money]."""
    out = []
    if action == RUN:
        out.append([_KIT_BREAKDOWN[state], FAILING, _KIT_PROFIT[state] - _KIT_TOW])
        for d in range(3):
            p = (1 - _KIT_BREAKDOWN[state]) * _KIT_WEAR[state][d]
            if p > 0:
                out.append([p, min(FAILING, state + d), _KIT_PROFIT[state]])
    elif action == SERVICE:
        for d in range(3):
            p = _KIT_REPAIR[state][d]
            if p > 0:
                out.append([p, max(HEALTHY, state - d), -_KIT_SHOP])
    else:
        out.append([1, HEALTHY, -_KIT_NEW_VAN])
    running = 0
    for o in out:
        running = running + o[0]
        o[0] = running
    return out


_KIT_WEEK = []
for _s in range(4):
    _KIT_WEEK.append([])
    for _a in range(3):
        _KIT_WEEK[_s].append(_kit_outcomes(_s, _a))

# The luck: one stream of numbers between 0 and 1 (Mulberry32, the generator of the arcade and of page 3)
_KIT_LUCK = {"seed": _KIT_FIRST_LUCK, "state": _KIT_FIRST_LUCK, "update": 0}


def _kit_next():
    s = (_KIT_LUCK["state"] + 0x6D2B79F5) & 0xFFFFFFFF
    _KIT_LUCK["state"] = s
    t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
    t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
    return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    return _kit_next()


def transition(state, action):
    """One week of Old Bessie, the transition dynamics of page 1. In this state, with this action, luck decides what
    happens, with the chances of page 1. Gives the van's state next week and the week's money:
        next_state, money = transition(HEALTHY, RUN)"""
    if state is Ellipsis or action is Ellipsis:
        print("⏳ transition waits for a state and an action: replace each ... with yours, then run the cell again.")
        return None, None
    try:
        outcomes = _KIT_WEEK[state][action]
    except (IndexError, TypeError):
        raise ValueError("transition needs a state from 0 to 3 and an action from 0 to 2; it got the state %r and "
                         "the action %r" % (state, action))
    u = _kit_next()
    for running, next_state, money in outcomes:
        if u < running:
            return next_state, money
    return outcomes[-1][1], outcomes[-1][2]


def same_luck():
    """The luck starts again from its beginning: the runs that follow meet the same weeks as the runs of every
    average before them, so two tables of bets are compared on the same luck."""
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]


def new_luck():
    """New runs for the next update of proto-REINFORCE: the luck starts from a new seed, 2000 plus the number of the
    update."""
    _KIT_LUCK["update"] = _KIT_LUCK["update"] + 1
    _KIT_LUCK["seed"] = _KIT_LUCK_BASE + _KIT_LUCK["update"]
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]


def _kit_set_luck(seed, update=0):
    """For the checks: the luck of this seed, from its beginning."""
    _KIT_LUCK["seed"] = seed
    _KIT_LUCK["state"] = seed
    _KIT_LUCK["update"] = update


def _kit_before_long():
    """play calls this before the loop: page 3's start, and the same picks for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_after_long():
    """play calls this after the loop: page 3's start again, for the cells above."""
    _kit_set_luck(_KIT_FIRST_LUCK)


def _kit_seed_picks(seed):
    """For the checks: Python's random from this seed."""
    random.seed(seed)


def _kit_save():
    """For the checks: the state of the luck and of Python's random."""
    return [dict(_KIT_LUCK), random.getstate()]


def _kit_restore(saved):
    """For the checks: the luck and Python's random as they were."""
    _KIT_LUCK.update(saved[0])
    random.setstate(saved[1])


def start_bets():
    """Page 3's table of bets picked at random, between -10 and 10: a fresh copy each time."""
    return copy_bets(_KIT_START_BETS)


def copy_bets(bets):
    """A copy of a table of bets: change the copy, and the table stays as it was."""
    copy = []
    for row in bets:
        copy.append(list(row))
    return copy


def _kit_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _kit_is_table(table, columns=3):
    if not isinstance(table, list) or len(table) != len(STATES):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != columns:
            return False
        for x in row:
            if not _kit_is_number(x):
                return False
    return True


def _kit_short(value):
    text = repr(value).replace("Ellipsis", "...")
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, places, mark):
    head = " " * 9
    for name in ACTIONS:
        head += "%10s" % name
    if mark:
        head += "    likeliest"
    print(head)
    for s in range(len(STATES)):
        line = "%-9s" % STATES[s]
        best = 0
        for a in range(len(ACTIONS)):
            line += "%10.*f" % (places, table[s][a])
            if table[s][a] > table[s][best]:
                best = a
        if mark:
            line += "    " + ACTIONS[best]
        print(line)


def show_bets(bets):
    """Print a table of bets: a row per state, a column per action, two decimals. Silent for None (a line above
    says why)."""
    if bets is None:
        return
    if not _kit_is_table(bets):
        print("⏳ No table of bets to show yet: show_bets needs four rows of three numbers. It got %s." % _kit_short(bets))
        return
    _kit_print_table(bets, 2, False)


def show_chances(table):
    """Print a probability table (a policy): a row per state, a column per action, two decimals, and the likeliest
    action of each row. Silent for None (a line above says why)."""
    if table is None:
        return
    if not _kit_is_table(table):
        print("⏳ No probability table to show yet: show_chances needs four rows of three numbers. It got %s."
              % _kit_short(table))
        return
    _kit_print_table(table, 2, True)


def show_counts(counts):
    """Print a table of counts: a row per state, a column per action, whole numbers."""
    if counts is None:
        return
    if not _kit_is_table(counts):
        print("⏳ No counts to show yet: show_counts needs four rows of three numbers. It got %s." % _kit_short(counts))
        return
    _kit_print_table(counts, 0, False)


def show_roulette(chances, ball=None):
    """Print the roulette of a row of chances: each action's slot, starting where the slot before it ends, on a wheel
    of 50 marks from 0 to 1. With a ball, a ^ marks where it is."""
    ok = isinstance(chances, list) and len(chances) == len(ACTIONS)
    if ok:
        for x in chances:
            if not _kit_is_number(x) or x < 0:
                ok = False
    if not ok:
        print("⏳ No roulette to show yet: show_roulette needs a row of three chances. It got %s." % _kit_short(chances))
        return
    start = 0
    for a in range(len(ACTIONS)):
        end = start + chances[a]
        left = int(round(start * 50))
        right = int(round(end * 50))
        print("%-8s %4.2f to %4.2f  |%s%s" % (ACTIONS[a], start, end, " " * left, "#" * (right - left)))
        start = end
    print(" " * 24 + "0" + " " * 49 + "1")
    if ball is not None and _kit_is_number(ball) and 0 <= ball < 1:
        print(" " * 24 + " " * int(ball * 50) + "^ the ball, %.2f" % ball)


def _kit_dollars(x):
    """$95, -$185, $119.30: whole dollars without cents."""
    if x == int(x):
        text = "$%d" % abs(int(x))
    else:
        text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def show_week(next_state, money):
    """Print one week of the van: its state next week and the week's money. Silent when there is no week yet (a
    line above says why)."""
    if next_state is None and money is None:
        return
    if isinstance(next_state, bool) or not isinstance(next_state, int) or not 0 <= next_state < len(STATES) or (
            not _kit_is_number(money)):
        print("⏳ No week to show yet: show_week needs a state and the week's money. It got %s and %s."
              % (_kit_short(next_state), _kit_short(money)))
        return
    print("Next week the van is %s. The week's money: %s." % (STATES[next_state], _kit_dollars(money)))


def show_money(label, money):
    """Print an amount of money, or a list of amounts, after a label: show_money("G1:", 154.4) prints G1: $154.40.
    A list longer than a run shows its first WEEKS. Silent when there is nothing to show yet (a line above says why)."""
    if money is None:
        return
    if isinstance(money, list):
        parts = []
        for k in range(len(money)):
            if k == WEEKS:
                parts.append("... (%s in all)" % format(len(money), ","))
                break
            if _kit_is_number(money[k]):
                parts.append(_kit_dollars(money[k]))
            else:
                parts.append(_kit_short(money[k]))
        print(label, ", ".join(parts) if parts else "none")
    elif _kit_is_number(money):
        print(label, _kit_dollars(money))
    else:
        print(label, _kit_short(money))


def show_value(label, value):
    """Print a value after a label: show_value("nabla =", 174). Silent for None (a line above says why)."""
    if value is None:
        return
    print(label, _kit_short(value))


def _kit_money(x):
    """-$172.24, $73.57"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def _kit_axis(x):
    """-$150, $250"""
    text = "$%d" % abs(int(round(x)))
    return "-" + text if x < 0 else text


def plot_rewards(points, best=_KIT_BEST_LINE, label="best strategy"):
    """The reward plot of page 3, as text: across, the number of updates; up, the average G1 after them. o is a
    point (one column for every few updates when there are many), and the row of dashes marks the best strategy's
    average G1 on the luck of the last update. Silent for None (a line above says why)."""
    if points is None:
        return
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if not _kit_is_number(x):
                ok = False
    if not ok:
        print("⏳ No reward plot yet: plot_rewards needs a list of at least two averages. It got %s." % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 10
    for size in (10, 25, 50, 100, 250, 500, 1000):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(high / step) * step
    bottom = _kit_math.floor(low / step) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The reward plot: the average G1 after each update" + (" (one o for every %d updates)" % every if every > 1
                                                                 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_axis(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 50 if n >= 200 else (10 if n >= 40 else 1)
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))


print("✅ Old Bessie's kit is ready.")

In [ ]:
#@title The checks of notebook 1 (run me too) { display-mode: "form" }
# The checks of notebook 1 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N1_TESTS = {'make_roulette': [{'about': "HEALTHY's row of my_table, [0.6, 0.3, 0.1]", 'args': [[0.6, 0.3, 0.1]], 'kind': 'numbers', 'want': [0.6, 0.8999999999999999, 0.9999999999999999], 'tol': 1e-09, 'keep': [0]}, {'about': "SHAKY's row of my_table, [0.2, 0.5, 0.3]", 'args': [[0.2, 0.5, 0.3]], 'kind': 'numbers', 'want': [0.2, 0.7, 1.0], 'tol': 1e-09, 'keep': [0]}, {'about': 'the row [0, 1, 0]', 'args': [[0, 1, 0]], 'kind': 'numbers', 'want': [0, 1, 1], 'tol': 1e-09}, {'about': 'a row of four chances, [0.25, 0.25, 0.25, 0.25]', 'args': [[0.25, 0.25, 0.25, 0.25]], 'kind': 'numbers', 'want': [0.25, 0.5, 0.75, 1.0], 'tol': 1e-09}], 'where_it_lands': [{'about': 'the ends [0.6, 0.9, 1.0] and a ball at 0.75', 'args': [[0.6, 0.9, 1.0], 0.75], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0]}, {'about': 'the ends [0.6, 0.9, 1.0] and a ball at 0.3', 'args': [[0.6, 0.9, 1.0], 0.3], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0]}, {'about': 'the ends [0.6, 0.9, 1.0] and a ball at 0.95', 'args': [[0.6, 0.9, 1.0], 0.95], 'kind': 'action', 'want': 2, 'tol': 0, 'keep': [0]}, {'about': 'the ends [0.2, 0.7, 1.0] and a ball at 0.2', 'args': [[0.2, 0.7, 1.0], 0.2], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0]}, {'about': 'the ends [0.2, 0.7, 1.0] and a ball at 0.0', 'args': [[0.2, 0.7, 1.0], 0.0], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0]}, {'about': 'the ends [0, 1, 1] and a ball at 0.5', 'args': [[0, 1, 1], 0.5], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0]}, {'about': 'the ends [0.25, 0.5, 0.75, 1.0] and a ball at 0.6', 'args': [[0.25, 0.5, 0.75, 1.0], 0.6], 'kind': 'action', 'want': 2, 'tol': 0, 'keep': [0]}], 'pick_action': [{'about': 'my_table in HEALTHY, with the ball at 0.2882', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 0], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table in HEALTHY, with the ball at 0.7343', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 0], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [2]]]}, {'about': 'my_table in HEALTHY, with the ball at 0.9236', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 0], 'kind': 'action', 'want': 2, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [4]]]}, {'about': 'my_table in WORN, with the ball at 0.6271', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 1], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [1]]]}, {'about': 'my_table in SHAKY, with the ball at 0.1563', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 2], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [8]]]}, {'about': 'my_table in FAILING, with the ball at 0.4480', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 3], 'kind': 'action', 'want': 2, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [14]]]}, {'about': "page 3's starting chances in HEALTHY, with the ball at 0.2882", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], 0], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}], 'five_weeks': [{'about': 'my_table, on luck 1 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]], 'kind': 'numbers', 'want': [95, 95, -240, -130, 95], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table, on luck 2 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]], 'kind': 'numbers', 'want': [-50, 95, -50, 95, 95], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [3]]]}, {'about': 'my_table, on luck 3 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]], 'kind': 'numbers', 'want': [95, -130, 95, 72, -50], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [7]]]}, {'about': 'my_table, on luck 4 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]], 'kind': 'numbers', 'want': [-50, 95, 72, -130, -130], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [5]]]}, {'about': 'a policy that always runs the van, [1, 0, 0] in every row, on luck 5 of the check', 'args': [[[1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0]]], 'kind': 'numbers', 'want': [95, 72, 40, 16, 16], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [9]]]}], 'total_of_five': [{'about': 'the money [95, 95, -240, -130, 95]', 'args': [[95, 95, -240, -130, 95]], 'kind': 'number', 'want': -85, 'tol': 1e-09, 'keep': [0]}, {'about': 'the money [-50, 95, -50, 95, 95]', 'args': [[-50, 95, -50, 95, 95]], 'kind': 'number', 'want': 185, 'tol': 1e-09, 'keep': [0]}, {'about': 'the money [95, -130, 95, 72, -50]', 'args': [[95, -130, 95, 72, -50]], 'kind': 'number', 'want': 82, 'tol': 1e-09, 'keep': [0]}], 'play_run': [{'about': 'my_table and 13 weeks, on luck 1 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 13], 'kind': 'numbers', 'want': [95, 95, -240, -130, 95, 72, -50, -130, 95, -130, 95, 72, -50], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table and 13 weeks, on luck 2 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 13], 'kind': 'numbers', 'want': [-130, 95, 95, 72, -130, -50, -50, 95, 72, -130, -130, 95, 72], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [4]]]}, {'about': 'my_table and 3 weeks, on luck 3 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 3], 'kind': 'numbers', 'want': [-185, -130, 95], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [6]]]}, {'about': 'a policy that always runs the van, [1, 0, 0] in every row and 13 weeks, on luck 4 of the check', 'args': [[[1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0]], 13], 'kind': 'numbers', 'want': [95, 72, 40, -240, -264, -264, -264, 16, -264, -264, -264, 16, -264], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [8]]]}, {'about': 'my_table and 1 week, on luck 5 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 1], 'kind': 'numbers', 'want': [95], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [10]]]}], 'cumulative_reward': [{'about': "page 2's run, [95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]", 'args': [[95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]], 'kind': 'number', 'want': 154.39614434222, 'tol': 1e-09, 'keep': [0]}, {'about': 'the rewards [100, 100]', 'args': [[100, 100]], 'kind': 'number', 'want': 190.0, 'tol': 1e-09, 'keep': [0]}, {'about': 'the rewards [10, -20, 30]', 'args': [[10, -20, 30]], 'kind': 'number', 'want': 16.3, 'tol': 1e-09, 'keep': [0]}, {'about': 'a run of no weeks, []', 'args': [[]], 'kind': 'number', 'want': 0, 'tol': 1e-09}], 'play_runs': [{'about': 'my_table, 4 runs of 3 weeks, on luck 1 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 3, 4], 'kind': 'table', 'want': [[95, 95, -240], [-50, 95, 72], [95, -50, 95], [-130, 95, 72]], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table, 2 runs of 13 weeks, on luck 2 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 13, 2], 'kind': 'table', 'want': [[-50, 95, 72, -130, -130, -50, -50, 95, -50, -50, -185, -130, 95], [95, -50, -130, 95, -50, -50, -50, 95, -50, -50, 95, -50, -130]], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [5]]]}, {'about': 'a policy that always runs the van, [1, 0, 0] in every row, 3 runs of 2 weeks, on luck 3 of the check', 'args': [[[1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0]], 2, 3], 'kind': 'table', 'want': [[95, 95], [95, 72], [95, 72]], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [7]]]}, {'about': 'my_table, 0 runs of 5 weeks, on luck 4 of the check', 'args': [[[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]], 5, 0], 'kind': 'table', 'want': [], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [1]]]}], 'cumulative_rewards': [{'about': 'the runs [[95, 72], [-185, 95]]', 'args': [[[95, 72], [-185, 95]]], 'kind': 'numbers', 'want': [159.8, -99.5], 'tol': 1e-09, 'keep': [0]}, {'about': "the runs [page 2's run]", 'args': [[[95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]]], 'kind': 'numbers', 'want': [154.39614434222], 'tol': 1e-09, 'keep': [0]}, {'about': 'the runs []', 'args': [[]], 'kind': 'numbers', 'want': [], 'tol': 1e-09, 'keep': [0]}], 'average': [{'about': 'the numbers [1, 2, 3, 4]', 'args': [[1, 2, 3, 4]], 'kind': 'number', 'want': 2.5, 'tol': 1e-09, 'keep': [0]}, {'about': 'the numbers [-172.24]', 'args': [[-172.24]], 'kind': 'number', 'want': -172.24, 'tol': 1e-09, 'keep': [0]}, {'about': 'the numbers [10, -10, 3]', 'args': [[10, -10, 3]], 'kind': 'number', 'want': 1.0, 'tol': 1e-09, 'keep': [0]}, {'about': 'the numbers [154.3961, 0]', 'args': [[154.39614434222, 0]], 'kind': 'number', 'want': 77.19807217111, 'tol': 1e-09, 'keep': [0]}], 'average_G1': [{'about': "page 3's bets, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -172.24, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets with the bet of HEALTHY and RUN one hop higher, on page 3's luck", 'args': [[[0.41000000000000003, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -170.5, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "the bets of warm-up 4, on page 3's luck", 'args': [[[0.4, -0.6, 5], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -945.97, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, on luck 2 of the check", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]], 'kind': 'number', 'want': -156.92, 'tol': 0.010000001, 'keep': [0], 'setup': [['_kit_set_luck', [2001]]]}], 'pick_bet': [{'about': 'pick 1 of the check', 'args': [], 'kind': 'numbers', 'want': [1, 2], 'tol': 0, 'setup': [['_kit_seed_picks', [1]]]}, {'about': 'pick 2 of the check', 'args': [], 'kind': 'numbers', 'want': [0, 0], 'tol': 0, 'setup': [['_kit_seed_picks', [2]]]}, {'about': 'pick 3 of the check', 'args': [], 'kind': 'numbers', 'want': [2, 2], 'tol': 0, 'setup': [['_kit_seed_picks', [5]]]}, {'about': 'pick 4 of the check', 'args': [], 'kind': 'numbers', 'want': [1, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [8]]]}, {'about': 'pick 5 of the check', 'args': [], 'kind': 'numbers', 'want': [2, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [13]]]}, {'about': 'pick 6 of the check', 'args': [], 'kind': 'numbers', 'want': [1, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [21]]]}], 'slope': [{'about': "page 3's bets, HEALTHY and RUN, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 0.01], 'kind': 'number', 'want': 174, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, WORN and SERVICE, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 1, 1, 0.01], 'kind': 'number', 'want': 6, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, FAILING and SERVICE, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 3, 1, 0.01], 'kind': 'number', 'want': -3, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, HEALTHY and RUN, on luck 2 of the check", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 0.01], 'kind': 'number', 'want': 224, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [2001]]]}], 'update_bet': [{'about': "page 3's update 1, HEALTHY and RUN with nabla 174 and alpha 0.01", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 174, 0.01], 'kind': 'number', 'want': 2.14, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}, {'about': "page 3's update 2, HEALTHY and REPLACE with nabla -13 and alpha 0.01", 'args': [[[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 2, -13, 0.01], 'kind': 'number', 'want': -0.03, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[2.14, -0.6, -0.03], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}, {'about': "page 3's bets, SHAKY and REPLACE with nabla 0 and alpha 0.01", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 2, 2, 0, 0.01], 'kind': 'number', 'want': 10.0, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10.0], [0.4, 0.6, -0.4]]}}, {'about': "page 3's bets, WORN and SERVICE with nabla 6 and alpha 0.02", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 1, 1, 6, 0.02], 'kind': 'number', 'want': -0.68, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -0.6, 0.1], [-0.7, -0.68, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}], 'proto_reinforce': [{'about': "page 3's bets, 3 updates with hop 0.01 and alpha 0.01, on the luck and the picks of the loop cell", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 3, 0.01, 0.01], 'kind': 'numbers', 'want': [-172.24, 123.82, 127.4, 119.29], 'tol': 0.010000001, 'table': {'arg': 0, 'want': [[2.64, -0.54, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}, 'setup': [['_kit_before_long', []]]}, {'about': "page 3's bets, 2 updates with hop 0.01 and alpha 0.01, on the luck and other picks", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 2, 0.01, 0.01], 'kind': 'numbers', 'want': [-172.24, -146.38, -155.82], 'tol': 0.010000001, 'table': {'arg': 0, 'want': [[0.4, -1.06, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.14]]}, 'setup': [['_kit_before_long', []], ['_kit_seed_picks', [10]]]}, {'about': "page 3's bets, 2 updates with hop 0.01 and alpha 0.02, on the luck and the picks of the loop cell", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 2, 0.01, 0.02], 'kind': 'numbers', 'want': [-172.24, 169.88, 169.23], 'tol': 0.010000001, 'table': {'arg': 0, 'want': [[4.88, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}, 'setup': [['_kit_before_long', []]]}]}
_N1_SLIPS = {'make_roulette': [{'answers': [[0.6, 0.3, 0.1], [0.2, 0.5, 0.3], [0, 1, 0], [0.25, 0.25, 0.25, 0.25]], 'hint': 'append running, where this slot ends, and not the chance itself'}, {'answers': [[0, 0.6, 0.8999999999999999], [0, 0.2, 0.7], [0, 0, 1], [0, 0.25, 0.5, 0.75]], 'hint': 'add the chance to running first, then append running: the first slot ends at its chance'}, {'answers': [0.9999999999999999, 1.0, 1, 1.0], 'hint': 'make_roulette gives the list of the ends: append running inside the loop, and return ends'}], 'where_it_lands': [{'answers': [1, 0, 2, 0, 0, 1, 2], 'hint': 'a ball exactly at the end of a slot is not below it, so it lands in the next slot: use <'}, {'answers': [0, 2, 0, 2, 2, 0, 0], 'hint': 'the ball lands in the first slot whose end is above the ball: ball < ends[slot]'}, {'answers': [1, 1, 1, 1, 1, 1, 1], 'hint': "compare the ball with the end of the slot, ends[slot], and not with the slot's number"}, {'answers': [0.9, 0.6, 1.0, 0.7, 0.2, 1, 0.75], 'hint': "give back the slot's number, slot, and not its end"}], 'pick_action': [{'answers': [0, 1, 2, 1, 0, 0, 0], 'hint': "take the row of the van's state, policy[state]"}, {'answers': [0, 0, 0, 0, 1, 2, 0], 'hint': 'throw the ball once: the given line ball = spin() does it, so land the ball you have, ball'}, {'answers': [[0.6, 0.8999999999999999, 0.9999999999999999], [0.6, 0.8999999999999999, 0.9999999999999999], [0.6, 0.8999999999999999, 0.9999999999999999], [0.5, 0.9, 1.0], [0.2, 0.7, 1.0], [0.1, 0.4, 1.0], [0.47422635216524245, 0.6486844775885634, 1.0]], 'hint': 'give back where the ball lands, where_it_lands(ends, ball)'}, {'answers': [0, 2, 2, 2, 0, 2, 0], 'hint': 'the ball lands on the ends of the slots, the list that make_roulette gives'}], 'five_weeks': [{'answers': [[95, 95, -185, -130, 95], [-50, 95, -50, 95, 95], [95, -130, 95, 72, -50], [-50, 95, 95, -50, -130], [95, 95, 95, 72, 95]], 'hint': 'each week starts from the state that the week before gave: transition(state, action)'}, {'answers': [[95, 95, -240, -130, 95], [-50, 95, -50, 95, 95], [95, -130, 95, 72, 72], [-50, 95, 72, -130, -130], [95, 72, 40, 16, 16]], 'hint': "the policy picks each week's action in that week's state: pick_action(policy, state)"}, {'answers': [[95, -130, -50, 95, -130], [-50, 95, 72, 72, 40], [95, -50, 95, 95, 72], [-50, -50, -50, -130, -50], [95, 72, 40, 16, -264]], 'hint': "the line above already picked the week's action: call transition(state, action), so that each week throws one ball"}], 'total_of_five': [{'answers': [95, 95, -50], 'hint': "add each week's money to the total so far: total = total + money[2]"}, {'answers': [-530, -105, 250], 'hint': 'each line adds the next week: money[2] for week 3, money[3] for week 4, money[4] for week 5'}], 'play_run': [{'answers': [[95, 95, -185, -130, 95, 95, -50, -50, 95, -130, 95, 95, 95], [-130, 95, 95, 95, -50, -50, -50, 95, 95, -50, -130, 95, 95], [-185, -130, 95], [95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95], [95]], 'hint': 'each week starts from the state that the week before gave: transition(state, action)'}, {'answers': [[95, 95, -240, -50, 40, 16, -264, -50, 40, -130, 95, 72, -240], [-130, 95, 95, 72, -50, -50, -50, 72, 40, -50, -130, 95, 72], [-185, -50, -264], [95, 72, 40, -240, -264, -264, -264, 16, -264, -264, -264, 16, -264], [95]], 'hint': "the policy picks each week's action in that week's state: pick_action(policy, state)"}, {'answers': [[0, 2, 3, 0, 1, 3, 3, 0, 1, 0, 1, 2, 2], [0, 0, 1, 3, 0, 0, 0, 1, 2, 0, 0, 1, 2], [3, 0, 0], [1, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2]], 'hint': "a reward is the week's money: rewards.append(money)"}], 'cumulative_reward': [{'answers': [184, 200, 20, 0], 'hint': "each week's money counts its weight, and the weight shrinks after every week: G1 = G1 + weight * money, then weight = weight * GAMMA"}, {'answers': [175.10000000000002, 190.0, 19.0, 0], 'hint': 'multiply the weight by GAMMA: weight = weight * GAMMA, so week 3 counts 0.9 x 0.9 = 0.81 times'}, {'answers': [11.297181459240006, 90.0, 24.3, 0], 'hint': 'add to what G1 already holds: G1 = G1 + weight * money'}], 'play_runs': [{'answers': [[[95, 95, -240, -130], [95, 72, -50, -130], [95, -130, 95, 72], [95, 95, 95, 95]], [[-50, 95], [95, -50]], [[95, 95, 72], [95, 95, 72], [95, 72, 40]], []], 'hint': 'each run lasts weeks weeks: play_run(policy, weeks)'}, {'answers': [[-13.900000000000006, 93.82000000000001, 126.95, 13.820000000000007], [-216.2658123945051, -89.68464718753006], [180.5, 159.8, 159.8], []], 'hint': 'keep the whole run, its list of rewards: runs.append(run)'}], 'cumulative_rewards': [{'answers': [[[95, 72], [-185, 95]], [[95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]], []], 'hint': "append the run's G1, cumulative_reward(run), and not the run itself"}], 'average': [{'answers': [10, -172.24, 3, 154.39614434222], 'hint': 'divide the total by how many numbers there are: total / len(numbers)'}, {'answers': [1.0, -172.24, 1.0, 0.0], 'hint': 'add x to the total so far: total = total + x'}], 'average_G1': [{'answers': [-439.56, -429.19, -439.56, -426.18], 'hint': 'turn the bets into chances first: policy = probability_table(bets)'}, {'answers': [-160.86, -159.11, -932.64, -131.85], 'hint': 'page 3 averages RUNS = 1,000 runs: play_runs(policy, WEEKS, RUNS)'}, {'answers': [-215.06, -215.06, -1268.47, -226.42], 'hint': 'play_runs takes the weeks of a run first, then how many runs: play_runs(policy, WEEKS, RUNS)'}], 'pick_bet': [{'answers': [(0, 2), (0, 0), (2, 1), (0, 1), (1, 1), (0, 1)], 'hint': 'a row is one of the 4 states: random.randint(0, 3) gives 0, 1, 2 or 3'}, {'answers': [(2, 2), (1, 0), (3, 2), (2, 1), (3, 1), (2, 1)], 'hint': 'the rows count from 0: random.randint(0, 3) gives 0, 1, 2 or 3'}, {'answers': [(1, 0), (0, 0), (2, 2), (1, 2), (2, 2), (1, 3)], 'hint': 'a column is one of the 3 actions: random.randint(0, 2) gives 0, 1 or 2'}], 'slope': [{'answers': [-9244, -364, -498, -8874], 'hint': 'raise the bet by the hop: hopped[state][action] + hop'}, {'answers': [0, 0, 0, 0], 'hint': 'After is the average G1 of the hopped bets: average_G1(hopped)'}, {'answers': [2, 0, 0, 2], 'hint': 'divide the change by the hop: (after - before) / hop'}, {'answers': [-174, -6, 3, -224], 'hint': 'the change is after - before: when After is higher, the slope is positive'}], 'update_bet': [{'answers': [-1.34, 0.23, 10.0, -0.92], 'hint': 'the average G1 has to go up, so the update adds: new_bet = w + alpha * nabla'}, {'answers': [174.4, -12.9, 10, 5.2], 'hint': 'alpha keeps the move small: new_bet = w + alpha * nabla'}, {'answers': [1.74, -0.13, 0.0, 0.12], 'hint': 'the new bet starts from the bet as it is: new_bet = w + alpha * nabla'}], 'proto_reinforce': [{'answers': [[-172.24, 224, 6, 0], [-172.24, -46, 26], [-172.24, 224, 0]], 'hint': 'a point is the average G1 after the update: points.append(average_G1(bets))'}, {'answers': [[-172.24, 123.82, 146.93, 157.62], [-172.24, 123.82, 146.93], [-172.24, 169.88, 169.7]], 'hint': 'each update picks its bet at random: state, action = pick_bet()'}, {'answers': [[-172.24, -156.92, -165.41, -155.91], [-172.24, -156.92, -165.41], [-172.24, -156.92, -165.41]], 'hint': 'each update moves its bet with your update_bet: update_bet(bets, state, action, nabla, alpha)'}, {'answers': [[-172.24, 119.29], [-172.24, -155.82], [-172.24, 169.23]], 'hint': 'one point after every update: points.append(average_G1(bets)) goes inside the loop'}, {'answers': [[-172.24, 123.82, 127.4, 119.29], [-172.24, -146.38, -155.82], [-172.24, 123.82, 127.4]], 'hint': 'the update moves the bet by alpha times nabla: update_bet(bets, state, action, nabla, alpha)'}]}
_N1_SUCCESS = {'make_roulette': "make_roulette works: HEALTHY's chances [0.6, 0.3, 0.1] give the ends [0.6, 0.9, 1.0].", 'where_it_lands': 'where_it_lands works: every ball lands in the right slot, at the ends of the slots too.', 'pick_action': 'pick_action works: every ball the check threw gave the action of its slot.', 'five_weeks': 'five_weeks works: each week starts from the state that the week before gave.', 'total_of_five': 'total_of_five works: [95, 95, -240, -130, 95] add up to -$85.', 'play_run': 'play_run works: on every luck of the check, your runs follow the van week by week.', 'cumulative_reward': "cumulative_reward works: page 2's run gives G1 = $154.40, as on page 2.", 'play_runs': 'play_runs works: the runs come back in a list, one list of rewards per run.', 'cumulative_rewards': 'cumulative_rewards works: one G1 per run, in the order of the runs.', 'average': 'average works: the average of [1, 2, 3, 4] is 2.5.', 'average_G1': "average_G1 works: page 3's bets give -$172.24 on page 3's luck, the first point of page 3's reward plot.", 'pick_bet': "pick_bet works: a row from 0 to 3 and a column from 0 to 2, as Python's random picked them.", 'slope': "slope works: page 3's update 1, HEALTHY and RUN, gives nabla = 174, as on page 3.", 'update_bet': "update_bet works: page 3's update 1 moves the bet of HEALTHY and RUN from 0.40 to 2.14.", 'proto_reinforce': 'proto_reinforce works: its first 3 updates give the points -$172.24, $123.82, $127.40, $119.29.'}
_N1_DEFAULT = {'make_roulette': 'add the chance to running, then append running to ends', 'where_it_lands': 'the ball lands in this slot when ball < ends[slot]', 'pick_action': 'chances is policy[state], ends is make_roulette(chances), and the action is where_it_lands(ends, ball)', 'five_weeks': 'each week is two lines: action = pick_action(policy, state), then state, money_2 = transition(state, action), with money_3 and money_5 in weeks 3 and 5', 'total_of_five': "each line adds the next week's money: total = total + money[2], then money[3], then money[4]", 'play_run': 'in the loop: action = pick_action(policy, state), then state, money = transition(state, action), then rewards.append(money)', 'cumulative_reward': 'G1 = G1 + weight * money, then weight = weight * GAMMA', 'play_runs': 'run = play_run(policy, weeks), then runs.append(run)', 'cumulative_rewards': 'G1s.append(cumulative_reward(run))', 'average': 'total = total + x in the loop, then return total / len(numbers)', 'average_G1': 'policy = probability_table(bets), then runs = play_runs(policy, WEEKS, RUNS), then G1s = cumulative_rewards(runs)', 'pick_bet': 'state = random.randint(0, 3), then action = random.randint(0, 2)', 'slope': 'hopped[state][action] + hop, then after = average_G1(hopped), then nabla = (after - before) / hop', 'update_bet': 'new_bet = w + alpha * nabla', 'proto_reinforce': 'state, action = pick_bet(), then nabla = slope(bets, state, action, hop), then update_bet(bets, state, action, nabla, alpha), then points.append(average_G1(bets))'}
_N1_KEEP_HINT = {'make_roulette': 'make_roulette reads the row of chances and leaves it as it is: the ends go into a new list', 'where_it_lands': 'where_it_lands reads the ends and leaves them as they are', 'pick_action': 'pick_action reads the probability table and leaves it as it is', 'five_weeks': 'five_weeks reads the probability table and leaves it as it is', 'total_of_five': 'total_of_five reads the list of money and leaves it as it is', 'play_run': 'play_run reads the probability table and leaves it as it is', 'cumulative_reward': 'cumulative_reward reads the rewards and leaves them as they are', 'play_runs': 'play_runs reads the probability table and leaves it as it is', 'cumulative_rewards': 'cumulative_rewards reads the runs and leaves them as they are', 'average': 'average reads the numbers and leaves them as they are', 'average_G1': 'average_G1 reads the table of bets and leaves it as it is: probability_table gives a new table', 'pick_bet': 'pick_bet takes no inputs', 'slope': 'slope leaves the table of bets as it is: the hop goes into the copy, hopped', 'update_bet': 'update_bet changes only the bet of state and action: bets[state][action] = round(new_bet, 2)', 'proto_reinforce': 'proto_reinforce changes the table of bets it gets: update_bet(bets, state, action, nabla, alpha) on that table, with no copy'}
_N1_TASKS = {'make_roulette': 'task A', 'where_it_lands': 'task B', 'pick_action': 'task C', 'five_weeks': 'task D', 'total_of_five': 'task E', 'play_run': 'task F', 'cumulative_reward': 'task G', 'play_runs': 'task H', 'cumulative_rewards': 'task I', 'average': 'task J', 'average_G1': 'task K', 'pick_bet': 'task L', 'slope': 'task M', 'update_bet': 'task N', 'proto_reinforce': 'task O'}
_N1_NEEDS = {'make_roulette': [], 'where_it_lands': [], 'pick_action': ['make_roulette', 'where_it_lands'], 'five_weeks': ['pick_action'], 'total_of_five': [], 'play_run': ['pick_action'], 'cumulative_reward': [], 'play_runs': ['play_run'], 'cumulative_rewards': ['cumulative_reward'], 'average': [], 'average_G1': ['play_runs', 'cumulative_rewards', 'average'], 'pick_bet': [], 'slope': ['average_G1'], 'update_bet': [], 'proto_reinforce': ['pick_bet', 'slope', 'update_bet', 'average_G1']}
_N1_LONG = ['proto_reinforce']
_N1_NONE_HINT = {}
_N1_DICE_OF = 'spin'
_N1_GAPS = {}
_N1_WORKING = {}
_N1_DICE = {}


def _n1_num(x):
    """A number with at most 4 decimals: 154.3961, 2.14, 174."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n1_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n1_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n1_num(value)


def _n1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n1_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n1_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n1_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n1_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n1_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n1_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n1_like(got, answer, tol):
    """True when got has the shape and the numbers of a slip's answer, within tol: a slip may give a number where a
    list is due (return total), or a list of numbers where a list of lists is due."""
    if isinstance(answer, (list, tuple)):
        if not isinstance(got, (list, tuple)) or len(got) != len(answer):
            return False
        for k in range(len(answer)):
            if not _n1_like(got[k], answer[k], tol):
                return False
        return True
    return _n1_is_number(got) and _n1_is_number(answer) and abs(got - answer) <= tol


def _n1_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n1_copy(x))
        return out
    return value


def _n1_dice(seed, first, count):
    """The dice of count runs of a seed from run first, rolled by the kit once per seed and kept."""
    rolled = _N1_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N1_DICE) > 3:
            _N1_DICE.clear()
        rolled = globals()[_N1_DICE_OF](seed, max(first + count, 10))
        _N1_DICE[seed] = rolled
    return rolled[first:first + count]


def _n1_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n1_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "run" in a:
            args.append(_n1_dice(a["run"][0], a["run"][1], 1)[0])
        else:
            args.append(_n1_copy(a))
    return args


def _n1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N1_GAPS:
        return _N1_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N1_GAPS[code] = gap
    return gap


def _n1_holds_gap(value):
    """True when a value, or a list inside it, still holds a ... (a gap of a cell above)."""
    if value is Ellipsis:
        return True
    if isinstance(value, (list, tuple)):
        for x in value:
            if _n1_holds_gap(x):
                return True
    return False


def _n1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n1_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N1_NEEDS.get(name, []):
        key.extend(_n1_key(other, seen + (name,)))
    return tuple(key)


def _n1_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n1_call.__code__.co_filename, globals()[_N1_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n1_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back (is that function finished, with its return line?), or a name holds None'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a state or an action, such as policy[state] for the row of a state and row[action] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = (update_bet "
                "changes the table itself and gives back only the new bet)")
    if 'transition needs' in text:
        return 'the van got a state or an action it does not know: check what pick_action gives back'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n1_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint). The kit's state (its luck) is put back
    afterwards when the kit can save it."""
    save = globals().get("_kit_save")
    saved = save() if callable(save) else None
    try:
        return _n1_failure_of(name, fn)
    finally:
        if saved is not None:
            globals()["_kit_restore"](saved)


def _n1_failure_of(name, fn):
    tests = _N1_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n1_args(test)
        kept = _n1_copy(args)
        for setup in test.get("setup", []):
            globals()[setup[0]](*setup[1])
        run = _n1_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n1_explain(run))
        got = run["answer"]
        if not _n1_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N1_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N1_DEFAULT[name]
            for slip in _N1_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n1_like(got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n1_short(test["want"]),
                                                                    _n1_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n1_short(kept[i])), _N1_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n1_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of bets should be %s, and it is %s" % (
                    name, _n1_short(t["want"]), _n1_short(args[t["arg"]])), _N1_KEEP_HINT[name])
    return None


def _n1_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n1_has_gap(fn):
        return False
    for other in _N1_NEEDS.get(name, []):
        if _n1_uses(fn, other) and not _n1_works(other):
            return False
    if name not in _N1_TESTS:
        return True
    key = _n1_key(name)
    if key not in _N1_WORKING:
        _N1_WORKING[key] = _n1_failure(name, fn) is None
    return _N1_WORKING[key]


def _n1_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N1_TASKS:
        task = " (%s)" % _N1_TASKS[name]
    if _n1_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N1_NEEDS.get(name, []):
        if not _n1_uses(fn, other) or _n1_works(other):
            continue
        if other in _N1_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n1_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N1_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N1_NEEDS.get(other, []):
            if deeper in _N1_TASKS and not _n1_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N1_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n1_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n1_waits(name, fn, "this check"):
        return
    failure = _n1_failure(name, fn)
    if failure is None:
        print("✅ " + _N1_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n1_safe_check(name, fn):
    try:
        _n1_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_make_roulette(fn):
    """Task a: the ends of the slots of several rows of chances"""
    _n1_safe_check("make_roulette", fn)

def check_where_it_lands(fn):
    """Task b: the slot of several balls, at the ends of the slots too"""
    _n1_safe_check("where_it_lands", fn)

def check_pick_action(fn):
    """Task c: the action of several states and tables, each on a ball the check throws"""
    _n1_safe_check("pick_action", fn)

def check_five_weeks(fn):
    """Task d: five weeks of several policies, each on luck the check deals"""
    _n1_safe_check("five_weeks", fn)

def check_total_of_five(fn):
    """Task e: the total of several lists of five weeks' money"""
    _n1_safe_check("total_of_five", fn)

def check_play_run(fn):
    """Task f: runs of several policies and lengths, each on luck the check deals"""
    _n1_safe_check("play_run", fn)

def check_cumulative_reward(fn):
    """Task g: the G1 of several runs, page 2's run among them"""
    _n1_safe_check("cumulative_reward", fn)

def check_play_runs(fn):
    """Task h: several runs at once, each on luck the check deals"""
    _n1_safe_check("play_runs", fn)

def check_cumulative_rewards(fn):
    """Task i: the G1 of every run of several lists of runs"""
    _n1_safe_check("cumulative_rewards", fn)

def check_average(fn):
    """Task j: the average of several lists of numbers"""
    _n1_safe_check("average", fn)

def check_average_G1(fn):
    """Task k: the average G1 of several tables of bets, each on luck the check deals"""
    _n1_safe_check("average_G1", fn)

def check_pick_bet(fn):
    """Task l: the bet of several picks, each from a seed of Python's random that the check sets"""
    _n1_safe_check("pick_bet", fn)

def check_slope(fn):
    """Task m: the slope of several bets, each on luck the check deals"""
    _n1_safe_check("slope", fn)

def check_update_bet(fn):
    """Task n: the new bet of several updates, and the table of bets after each"""
    _n1_safe_check("update_bet", fn)

def check_proto_reinforce(fn):
    """Task o: the points and the table of bets after a few updates, on the luck and the picks the check sets"""
    _n1_safe_check("proto_reinforce", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_G1, bets) is
    average_G1(bets). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_G1, bets).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n1_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
            if _n1_holds_gap(x):
                print("⏳ %s waits for a value that still holds a ...: fill it in above, run its cell, then this cell."
                      % name)
                return None
        if name in _N1_LONG and not _n1_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        long = name in _N1_LONG
        if long and callable(globals().get("_kit_before_long")):
            globals()["_kit_before_long"]()
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        finally:
            if long and callable(globals().get("_kit_after_long")):
                globals()["_kit_after_long"]()
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

_N1_TABLE = [[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]
_N1_START_BETS = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]


def _n1_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N1_TASKS[name], cell))
        return False
    if _n1_waits(name, fn, cell):
        return False
    if not _n1_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def _n1_chances_ready(chances, cell):
    """True when chances is a row of three numbers; else one calm line."""
    if chances is None or _n1_holds_gap(chances):
        print("⏳ %s waits for a row of chances: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    if not isinstance(chances, list) or len(chances) != 3:
        print("⏳ %s needs a row of three chances, such as my_table[HEALTHY]." % cell)
        return False
    return True


def _n1_policy_ready(policy, cell):
    """True when policy is a probability table without gaps; else one calm line."""
    if policy is None or _n1_holds_gap(policy):
        print("⏳ %s waits for a probability table: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    return True


def check_table(table):
    """Warm-up 1: my_table, the made-up probability table."""
    try:
        if _n1_holds_gap(table):
            print("⏳ my_table is not finished yet (warm-up 1): replace each ... with a row of three chances, run its "
                  "cell, then this check again.")
            return
        if not isinstance(table, list) or len(table) != 4:
            print("❌ my_table should be a list of four rows, one per state, and yours is %s." % _n1_short(table))
            print("   💡 Hint: write one row per line, each in [ ] and followed by a comma, all inside the outer [ ].")
            return
        for s in range(4):
            row = table[s]
            if not _n1_same("numbers", row, _N1_TABLE[s], 1e-9):
                print("❌ %s's row should be %s, and yours is %s." % (STATES[s], _n1_text(_N1_TABLE[s]),
                                                                   _n1_short(row)))
                hint = "copy the row from the table above, three numbers in the order RUN, SERVICE, REPLACE"
                for other in range(4):
                    if other != s and _n1_same("numbers", row, _N1_TABLE[other], 1e-9):
                        hint = ("that is %s's row: the rows go in the order HEALTHY, WORN, SHAKY, FAILING"
                                % STATES[other])
                if isinstance(row, list) and len(row) == 3 and _n1_same("numbers", sorted(row), sorted(_N1_TABLE[s]),
                                                                        1e-9) and hint.startswith("copy"):
                    hint = "the numbers of a row go in the order of the columns: RUN, SERVICE, REPLACE"
                print("   💡 Hint: " + hint + ".")
                return
        print("✅ my_table is the made-up probability table: four rows, one per state, and each row adds up to 1.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_reading(chance, row):
    """Warm-up 2: a number and a row read from my_table."""
    try:
        if chance is Ellipsis or _n1_holds_gap(row):
            print("⏳ Warm-up 2 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        want_chance = _N1_TABLE[FAILING][REPLACE]
        want_row = _N1_TABLE[SHAKY]
        if not _n1_same("number", chance, want_chance, 1e-9):
            print("❌ chance should be %s, the chance of REPLACE when the van is FAILING, and yours is %s."
                  % (_n1_text(want_chance), _n1_short(chance)))
            hint = "the row comes first, then the column: my_table[FAILING][REPLACE]"
            if _n1_same("number", chance, _N1_TABLE[FAILING][SERVICE], 1e-9) or _n1_same(
                    "number", chance, _N1_TABLE[FAILING][RUN], 1e-9):
                hint = "that is another column of FAILING's row: REPLACE is the column REPLACE, number 2"
            print("   💡 Hint: " + hint + ".")
            return
        if not _n1_same("numbers", row, want_row, 1e-9):
            print("❌ row should be SHAKY's row, %s, and yours is %s." % (_n1_text(want_row), _n1_short(row)))
            print("   💡 Hint: a row is the table with one [ ]: my_table[SHAKY].")
            return
        print("✅ Both right: my_table[FAILING][REPLACE] is %s, and my_table[SHAKY] is %s." % (
            _n1_text(want_chance), _n1_text(want_row)))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_garage(next_state, money):
    """Warm-up 3: one week of a WORN van in the garage, transition(WORN, SERVICE)."""
    try:
        if next_state is None and money is None:
            print("⏳ Warm-up 3 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        for running, state, cash in _KIT_WEEK[WORN][SERVICE]:
            if next_state == state and money == cash:
                print("✅ A week of a WORN van in the garage: next week it is %s, and the week's money is %s." % (
                    STATES[next_state], _kit_dollars(money)))
                return
        print("❌ A WORN van in the garage costs %s for the week and comes back HEALTHY or WORN; your week gave the "
              "state %s and the money %s." % (_kit_dollars(_KIT_SHOP), _n1_short(next_state), _n1_short(money)))
        hint = "call transition with the state WORN first, then the action SERVICE: transition(WORN, SERVICE)"
        if money == -_KIT_NEW_VAN:
            hint = "that is the money of REPLACE: the action of the garage is SERVICE"
        elif _n1_is_number(money) and money != -_KIT_SHOP:
            hint = "that money comes from the route, the action RUN: the action of the garage is SERVICE"
        print("   💡 Hint: " + hint + ".")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def _n1_softmax(row):
    largest = max(row)
    powers = []
    for x in row:
        powers.append(exp(x - largest))
    total = 0
    for x in powers:
        total = total + x
    out = []
    for x in powers:
        out.append(x / total)
    return out


def check_my_bets(bets):
    """Warm-up 4: page 3's bets with the bet of HEALTHY and REPLACE set to 5."""
    try:
        if bets is None or _n1_holds_gap(bets):
            print("⏳ Warm-up 4 is not finished yet: replace the ... with your code, run its cell, then this check "
                  "again.")
            return
        if not _kit_is_table(bets):
            print("❌ my_bets should be a table of bets, four rows of three numbers, and it is %s." % _n1_short(bets))
            print("   💡 Hint: start from my_bets = start_bets(), then change one bet.")
            return
        changed = []
        for s in range(4):
            for a in range(3):
                if bets[s][a] != _N1_START_BETS[s][a]:
                    changed.append([s, a])
        if len(changed) == 0:
            print("⏳ my_bets is still page 3's table: set the bet of HEALTHY and REPLACE to 5 in the cell of warm-up "
                  "4, run it, then this check again.")
            return
        if changed == [[HEALTHY, REPLACE]] and bets[HEALTHY][REPLACE] == 5:
            print("✅ The bet of HEALTHY and REPLACE is 5: HEALTHY's chance of REPLACE went from %.2f to %.2f." % (
                _n1_softmax(_N1_START_BETS[HEALTHY])[REPLACE], _n1_softmax(bets[HEALTHY])[REPLACE]))
            return
        if len(changed) > 1:
            print("❌ %d bets of my_bets differ from page 3's; warm-up 4 changes one." % len(changed))
            print("   💡 Hint: start again from my_bets = start_bets(), then change only my_bets[HEALTHY][REPLACE].")
            return
        s, a = changed[0]
        if [s, a] != [HEALTHY, REPLACE]:
            print("❌ You changed the bet of %s and %s; warm-up 4 changes the bet of HEALTHY and REPLACE." % (
                STATES[s], ACTIONS[a]))
            print("   💡 Hint: the row comes first, then the column: my_bets[HEALTHY][REPLACE] = 5.")
            return
        print("❌ The bet of HEALTHY and REPLACE is %s; warm-up 4 sets it to 5." % _n1_short(bets[s][a]))
        print("   💡 Hint: my_bets[HEALTHY][REPLACE] = 5.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def throw_balls(chances, throws):
    """Throws the ball throws times on the roulette of a row of chances, with your make_roulette and where_it_lands,
    and prints how often each action came."""
    try:
        if not _n1_chances_ready(chances, "throw_balls"):
            return
        if not _n1_ready("make_roulette", "this cell") or not _n1_ready("where_it_lands", "this cell"):
            return
        ends = make_roulette(chances)
        counts = [0, 0, 0]
        for k in range(throws):
            slot = where_it_lands(ends, spin())
            counts[slot] = counts[slot] + 1
        print("In %s throws:  RUN %d times, SERVICE %d times, REPLACE %d times" % (
            format(throws, ","), counts[0], counts[1], counts[2]))
        print("Their shares:     RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (
            counts[0] / throws, counts[1] / throws, counts[2] / throws))
        print("The chances:      RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (chances[0], chances[1], chances[2]))
    except Exception as error:
        print("❌ throw_balls could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_picks(policy, picks):
    """Your pick_action, picks times in each state of a probability table: how often each action came, and the
    shares."""
    try:
        if not _n1_policy_ready(policy, "show_picks"):
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        counts = []
        shares = []
        for s in range(4):
            counts.append([0, 0, 0])
            for k in range(picks):
                action = pick_action(policy, s)
                counts[s][action] = counts[s][action] + 1
            shares.append([counts[s][0] / picks, counts[s][1] / picks, counts[s][2] / picks])
        print("Your pick_action, %s times in each state: how often each action came" % format(picks, ","))
        show_counts(counts)
        print()
        print("Their shares, to set beside the probability table")
        show_chances(shares)
    except Exception as error:
        print("❌ show_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def count_picks(how_many):
    """Your pick_bet, how_many times: how often each bet came."""
    try:
        if not _n1_ready("pick_bet", "this cell"):
            return
        counts = [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
        for k in range(how_many):
            state, action = pick_bet()
            counts[state][action] = counts[state][action] + 1
        print("Your pick_bet, %s times: how often each bet came" % format(how_many, ","))
        show_counts(counts)
    except Exception as error:
        print("❌ count_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_run(policy, weeks):
    """Plays one run with your pick_action and the kit's transition, and prints it week by week."""
    try:
        if not _n1_policy_ready(policy, "show_run"):
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        state = HEALTHY
        print("week  state     action     money")
        for week in range(weeks):
            action = pick_action(policy, state)
            next_state, money = transition(state, action)
            print("%4d  %-8s  %-8s  %6s" % (week + 1, STATES[state], ACTIONS[action], _kit_dollars(money)))
            state = next_state
    except Exception as error:
        print("❌ show_run could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of notebook 1 are ready.")

## 1. The probability table

A strategy with chances is a probability table, as on page 3: a row for each state of the van, a column for each
action, and each row adds up to 1. Page 3 also calls it a policy. Here is a made-up one:

| | RUN | SERVICE | REPLACE |
|---|---|---|---|
| **HEALTHY** | 0.6 | 0.3 | 0.1 |
| **WORN** | 0.5 | 0.4 | 0.1 |
| **SHAKY** | 0.2 | 0.5 | 0.3 |
| **FAILING** | 0.1 | 0.3 | 0.6 |

When the van is WORN, this policy runs it in 5 weeks out of 10 on average, services it in 4 and replaces it
in 1.

In Python, a table is a list of rows, and each row is a list of three numbers in the order of the columns: RUN,
SERVICE, REPLACE. The rows come in the order HEALTHY, WORN, SHAKY, FAILING. HEALTHY's row is written for you.

### 🎯 Warm-up 1. Write the table in Python

In [ ]:
# 🎯 YOUR TURN, warm-up 1: the made-up probability table, one row per state
my_table = [
    [0.6, 0.3, 0.1],        # HEALTHY: RUN, SERVICE, REPLACE
    ...,                    # 🎯 YOUR TURN: WORN's row
    ...,                    # 🎯 YOUR TURN: SHAKY's row
    ...,                    # 🎯 YOUR TURN: FAILING's row
]
show_chances(my_table)

In [ ]:
#@title 💡 Hint 1 for warm-up 1 (double-click to open) { display-mode: "form" }
# Each ... becomes one row: three numbers in [ ], separated by commas, copied from the table above.
# Keep the comma after each row: the rows are the items of one list.

In [ ]:
#@title 💡 Hint 2 for warm-up 1 (double-click to open) { display-mode: "form" }
# WORN's row reads from the table: RUN 0.5, SERVICE 0.4, REPLACE 0.1. So the line is
#     [0.5, 0.4, 0.1],        # WORN

In [ ]:
# Check warm-up 1
check_table(my_table)

### Reading a table

The kit gives each state and each action its number, counted from 0: `HEALTHY` is 0, `WORN` 1, `SHAKY` 2,
`FAILING` 3, and `RUN` is 0, `SERVICE` 1, `REPLACE` 2. So `my_table[HEALTHY]` is HEALTHY's row, and
`my_table[HEALTHY][RUN]` is one number of that row: the chance of RUN when the van is HEALTHY. The row comes
first, then the column.

In [ ]:
# A row, then one number of it
print("HEALTHY's row:", my_table[HEALTHY])
print("The chance of RUN when the van is HEALTHY:", my_table[HEALTHY][RUN])

### 🎯 Warm-up 2. Read your table

Put the chance of REPLACE when the van is FAILING into `chance`, and SHAKY's row into `row`.

In [ ]:
# 🎯 YOUR TURN, warm-up 2: read your table
chance = ...                # 🎯 YOUR TURN: the chance of REPLACE when the van is FAILING
row = ...                   # 🎯 YOUR TURN: SHAKY's row

In [ ]:
#@title 💡 Hint 1 for warm-up 2 (double-click to open) { display-mode: "form" }
# A number of the table takes two [ ]: first the row, the state, then the column, the action.
# A row takes one [ ]: the state.

In [ ]:
#@title 💡 Hint 2 for warm-up 2 (double-click to open) { display-mode: "form" }
#     chance = my_table[FAILING][REPLACE]
#     row = my_table[SHAKY]

In [ ]:
# Check warm-up 2
check_reading(chance, row)

## 2. A roulette

To follow a probability table, the van needs an action picked at random with the chances of its row. A roulette
does it. Cut a wheel of length 1 into three slots, one for each action, each as long as its chance. HEALTHY's
row, 0.6, 0.3 and 0.1, cuts it like this: RUN from 0 to 0.6, SERVICE from 0.6 to 0.9, REPLACE from 0.9 to 1.

Then throw a ball. It lands somewhere between 0 and 1, every place as likely as any other, and the slot it lands
in is the action. RUN's slot is 0.6 long, so the ball lands in it in 60% of the throws.

In [ ]:
# HEALTHY's roulette: each action's slot, on a wheel from 0 to 1
show_roulette(my_table[HEALTHY])

The kit's `spin()` throws the ball: it gives a number between 0 and 1, every number as likely as any other. The
next cell throws one ball on HEALTHY's roulette and marks where it is. Run it a few times.

In [ ]:
# One ball on HEALTHY's roulette
ball = spin()
show_roulette(my_table[HEALTHY], ball)

### Where the slots end

Each slot ends where the slot before it ended, plus its own chance: RUN ends at 0.6, SERVICE at 0.6 + 0.3 = 0.9,
REPLACE at 0.9 + 0.1 = 1. These ends, a running total of the chances, make the roulette.

### 🎯 Task A. The ends of the slots

Write `make_roulette(chances)`: it walks along the row and keeps a running total, and after each chance it notes
where that slot ends.

In [ ]:
# 🎯 YOUR TURN, task A: the ends of the slots of a roulette
def make_roulette(chances):
    # chances: a row of chances, one per action.
    # Gives the ends of the slots: a running total of the chances, such as [0.6, 0.9, 1.0].
    ends = []
    running = 0                         # where the slots so far end: nowhere yet
    for chance in chances:
        running = ...                   # 🎯 YOUR TURN: this slot ends one chance further
        ends.append(...)                # 🎯 YOUR TURN: remember where this slot ends
    return ends

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# running starts at 0. Each chance moves it further: the new running is the old running plus the chance.
# After that, running is where this slot ends: put it into ends.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#     running = running + chance
#     ends.append(running)

In [ ]:
# Check task A
check_make_roulette(make_roulette)

### Where the ball lands

The ball lands in the first slot whose end is above it. With the ends 0.6, 0.9 and 1, a ball at 0.75 is not below
0.6, so it is not in RUN's slot; it is below 0.9, so it lands in SERVICE's slot.

💭 **Think first.** With SHAKY's ends, 0.2, 0.7 and 1, where does a ball at 0.71 land? And a ball at exactly 0.2?

In [ ]:
#@title The answer (double-click to open) { display-mode: "form" }
# A ball at 0.71 is not below 0.2 and not below 0.7; it is below 1, so it lands in REPLACE's slot.
# A ball at exactly 0.2 is not below 0.2, since it is equal; it is below 0.7, so it lands in SERVICE's slot. A
# slot holds its start and stops just before its end.

### 🎯 Task B. The slot where the ball lands

Write `where_it_lands(ends, ball)`: it looks at the slots in order and gives the first one whose end is above the
ball. A slot is a number: 0 for RUN's slot, 1 for SERVICE's, 2 for REPLACE's.

In [ ]:
# 🎯 YOUR TURN, task B: the slot where the ball lands
def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    for slot in range(len(ends)):
        if ...:                         # 🎯 YOUR TURN: the ball is below the end of this slot
            return slot
    return len(ends) - 1                # a ball at the very end of the wheel: the last slot

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# ends[slot] is where this slot ends. The ball is in this slot when it is below that end.
# Below is <. The loop goes through the slots in order, so the first slot that passes is the right one.

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
#     if ball < ends[slot]:

In [ ]:
# Check task B
check_where_it_lands(where_it_lands)

### Many throws

One throw can land anywhere. Many throws land in each slot about as often as its chance: the next cell throws the
ball 1,000 times on HEALTHY's roulette, with your two functions. Run it a few times.

In [ ]:
# 1,000 throws on HEALTHY's roulette
throw_balls(my_table[HEALTHY], 1000)

**Reading the result.** The shares are close to the chances and never exactly equal: a roulette follows its
chances on average, and each set of throws has its own luck.

## 3. Picking an action

Now put the two together. In a state, the van's action comes from the roulette of that state's row in the
probability table. A probability table is a policy, so the function calls it `policy`.

### 🎯 Task C. An action at random

Write `pick_action(policy, state)`: take the state's row, make its roulette, throw the ball once, and give the
slot where it lands.

In [ ]:
# 🎯 YOUR TURN, task C: an action at random, with the chances of the state's row
def pick_action(policy, state):
    # policy: a probability table, one row per state. state: the van's state, the number of a row.
    # Gives an action picked at random with the chances of that row: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    chances = ...                       # 🎯 YOUR TURN: the row of the state
    ends = ...                          # 🎯 YOUR TURN: its roulette, with your make_roulette
    ball = spin()                       # the ball
    return ...                          # 🎯 YOUR TURN: where the ball lands, with your where_it_lands

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The row of the state is policy[state]. Your make_roulette turns it into the ends of the slots.
# The ball is thrown for you. Your where_it_lands says where it lands, given the ends and the ball.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
#     chances = policy[state]
#     ends = make_roulette(chances)
#     return where_it_lands(ends, ball)

In [ ]:
# Check task C
check_pick_action(pick_action)

In [ ]:
# Your pick_action, 1,000 times in each state of my_table
show_picks(my_table, 1000)

**Reading the result.** Each row of shares is close to the same row of `my_table`: WORN's picks are about half
RUN, and FAILING's mostly REPLACE.

## 4. One week of the van

The van itself is in the kit: `transition(state, action)`, the transition dynamics of page 1. In a state, with an
action, luck decides what happens, with the chances of page 1, and `transition` gives back two things: the van's
state next week and the week's money. Two names on the left of `=` catch them:

```python
next_state, money = transition(HEALTHY, RUN)
```

The money of a week, as on page 1: RUN earns \$95, \$72, \$40 or \$16 by the van's wear, and a breakdown costs \$280 more;
SERVICE costs \$50; REPLACE costs \$130. Run the next cell a few times: the same state and action do not always give
the same week.

In [ ]:
# One week of a HEALTHY van on its route
next_state, money = transition(HEALTHY, RUN)
show_week(next_state, money)

### 🎯 Warm-up 3. One week in the garage

Call `transition` yourself, once: a WORN van, and the action SERVICE.

In [ ]:
# 🎯 YOUR TURN, warm-up 3: one week of a WORN van in the garage
next_state, money = transition(..., ...)        # 🎯 YOUR TURN: the state WORN, then the action SERVICE
show_week(next_state, money)

In [ ]:
#@title 💡 Hint 1 for warm-up 3 (double-click to open) { display-mode: "form" }
# transition takes two inputs: first the state, then the action. The kit's names WORN and SERVICE stand for
# their numbers.

In [ ]:
#@title 💡 Hint 2 for warm-up 3 (double-click to open) { display-mode: "form" }
#     next_state, money = transition(WORN, SERVICE)

In [ ]:
# Check warm-up 3
check_garage(next_state, money)

### Five weeks

A quarter of the van is a chain of such weeks. Each week, the policy picks an action in the van's state,
`transition` gives the next state and the money, and the next week starts from that state.

### 🎯 Task D. Five weeks, written out

Write `five_weeks(policy)`: five weeks from HEALTHY, each a call of `pick_action` and a call of `transition` with
the state that the week before gave. Weeks 1 and 4 are written for you.

In [ ]:
# 🎯 YOUR TURN, task D: five weeks of the van, written out
def five_weeks(policy):
    # policy: a probability table. Plays five weeks of the van from HEALTHY, one after the other.
    # Gives the money of the five weeks, in a list.
    state = HEALTHY
    action = pick_action(policy, state)             # week 1: the policy picks an action in the van's state
    state, money_1 = transition(state, action)      # week 1: the van's week; state is now next week's state
    action = ...                                    # 🎯 YOUR TURN: week 2, the policy's action
    state, money_2 = ...                            # 🎯 YOUR TURN: week 2, the van's week
    action = ...                                    # 🎯 YOUR TURN: week 3, the policy's action
    state, money_3 = ...                            # 🎯 YOUR TURN: week 3, the van's week
    action = pick_action(policy, state)             # week 4, the policy's action
    state, money_4 = transition(state, action)      # week 4, the van's week
    action = ...                                    # 🎯 YOUR TURN: week 5, the policy's action
    state, money_5 = ...                            # 🎯 YOUR TURN: week 5, the van's week
    return [money_1, money_2, money_3, money_4, money_5]

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Each week is the same two lines as week 1: the policy picks an action in the van's state, then transition
# plays the week from that state with that action.
# The name state on the left of = takes next week's state, so the next week starts from it.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
# Week 2:
#     action = pick_action(policy, state)
#     state, money_2 = transition(state, action)
# Weeks 3 and 5 are the same two lines, with money_3 and money_5.

In [ ]:
# Check task D
check_five_weeks(five_weeks)

### 🎯 Task E. The total of five weeks

On page 2, a run's cumulative reward was first the total of its rewards. Write `total_of_five(money)`: it adds
the five weeks' money one week at a time.

In [ ]:
# 🎯 YOUR TURN, task E: the total of five weeks
def total_of_five(money):
    # money: the money of five weeks, in a list. Gives their total.
    total = money[0]                    # week 1
    total = total + money[1]            # week 2 added
    total = ...                         # 🎯 YOUR TURN: week 3 added
    total = ...                         # 🎯 YOUR TURN: week 4 added
    total = ...                         # 🎯 YOUR TURN: week 5 added
    return total

In [ ]:
#@title 💡 Hint 1 for task E (double-click to open) { display-mode: "form" }
# Each line adds one more week's money to the total so far: the new total is the old total plus that week's
# money.
# money[2] is week 3's money: the places of a list count from 0.

In [ ]:
#@title 💡 Hint 2 for task E (double-click to open) { display-mode: "form" }
#     total = total + money[2]
#     total = total + money[3]
#     total = total + money[4]

In [ ]:
# Check task E
check_total_of_five(total_of_five)

In [ ]:
# Five weeks of my_table's policy and their total: run it a few times
money_of_five = play(five_weeks, my_table)
show_money("The five weeks' money:", money_of_five)
show_money("Their total:", play(total_of_five, money_of_five))

## 5. A run

A run is a quarter of the van: `WEEKS` = 13 weeks from HEALTHY, each week as in task D. Its rewards are the
weeks' money, in order. The next cell plays a run with your `pick_action` and prints it week by week. Run it a few
times.

In [ ]:
# A run of my_table's policy, week by week
show_run(my_table, WEEKS)

### 🎯 Task F. A run

Write `play_run(policy, weeks)`: the two lines of a week inside a loop that runs `weeks` times, and each week's
money appended to the list of rewards.

In [ ]:
# 🎯 YOUR TURN, task F: a run
def play_run(policy, weeks):
    # policy: a probability table. weeks: how many weeks the run lasts.
    # Plays a run of the van from HEALTHY. Gives its rewards: the money of every week, in a list.
    rewards = []
    state = HEALTHY
    for week in range(weeks):
        action = ...                    # 🎯 YOUR TURN: the policy picks an action in the van's state
        state, money = ...              # 🎯 YOUR TURN: the van's week
        rewards.append(...)             # 🎯 YOUR TURN: remember the week's money
    return rewards

In [ ]:
#@title 💡 Hint 1 for task F (double-click to open) { display-mode: "form" }
# The loop's body is one week of task D: the action from pick_action(policy, state), then the week from
# transition(state, action).
# rewards.append(...) keeps the week's money: the name next to state on the left of the transition line.

In [ ]:
#@title 💡 Hint 2 for task F (double-click to open) { display-mode: "form" }
#     action = pick_action(policy, state)
#     state, money = transition(state, action)
#     rewards.append(money)

In [ ]:
# Check task F
check_play_run(play_run)

### The cumulative reward

On page 2, later rewards then counted a little less: with $\gamma = 0.9$, week 1's money counts in full, week
2's 0.9 times, week 3's 0.81 times, and so on. With $r_1, r_2, r_3, \dots$ the weeks' money:

$$G_1 = r_1 + \gamma\, r_2 + \gamma^2\, r_3 + \dots$$

### 🎯 Task G. The cumulative reward of a run

Write `cumulative_reward(rewards)`: walk along the rewards with a weight that starts at 1 and is multiplied by
`GAMMA` after each week.

In [ ]:
# 🎯 YOUR TURN, task G: the cumulative reward of a run
def cumulative_reward(rewards):
    # rewards: the money of every week of a run, in a list.
    # Gives the run's cumulative reward G1: week 1's money counts in full, and each later week's money counts
    # GAMMA = 0.9 times as much as the week before.
    G1 = 0
    weight = 1                          # week 1 counts in full
    for money in rewards:
        G1 = ...                        # 🎯 YOUR TURN: add the week's money, times its weight
        weight = ...                    # 🎯 YOUR TURN: the next week counts GAMMA times as much
    return G1

In [ ]:
#@title 💡 Hint 1 for task G (double-click to open) { display-mode: "form" }
# Each week adds its money times its weight to G1. Then the weight shrinks: it is multiplied by GAMMA.
# Week 1 counts 1 time, week 2 0.9 times, week 3 0.9 x 0.9 = 0.81 times.

In [ ]:
#@title 💡 Hint 2 for task G (double-click to open) { display-mode: "form" }
#     G1 = G1 + weight * money
#     weight = weight * GAMMA

In [ ]:
# Check task G
check_cumulative_reward(cumulative_reward)

In [ ]:
# A run of my_table's policy and its cumulative reward: run it a few times
rewards = play(play_run, my_table, WEEKS)
show_money("The run's rewards:", rewards)
show_money("Its cumulative reward G1:", play(cumulative_reward, rewards))

## 6. Many runs and their average

One run says little: luck decides much of its $G_1$. Page 3 judged a policy by the average $G_1$ of `RUNS` =
1,000 runs. Three small functions get there: the runs, their $G_1$, their average.

### 🎯 Task H. Many runs

Write `play_runs(policy, weeks, how_many)`: `how_many` runs of the same policy, collected in a list.

In [ ]:
# 🎯 YOUR TURN, task H: many runs
def play_runs(policy, weeks, how_many):
    # Plays how_many runs of weeks weeks with the same policy.
    # Gives the runs in a list, each run the list of its rewards.
    runs = []
    for k in range(how_many):
        run = ...                       # 🎯 YOUR TURN: one run, with your play_run
        runs.append(...)                # 🎯 YOUR TURN: remember the run
    return runs

In [ ]:
#@title 💡 Hint 1 for task H (double-click to open) { display-mode: "form" }
# One run is play_run(policy, weeks): the same policy and the same number of weeks every time.
# runs.append(...) puts the run, its whole list of rewards, at the end of runs.

In [ ]:
#@title 💡 Hint 2 for task H (double-click to open) { display-mode: "form" }
#     run = play_run(policy, weeks)
#     runs.append(run)

In [ ]:
# Check task H
check_play_runs(play_runs)

### 🎯 Task I. The $G_1$ of every run

Write `cumulative_rewards(runs)`: the cumulative reward of each run, in a list.

In [ ]:
# 🎯 YOUR TURN, task I: the G1 of every run
def cumulative_rewards(runs):
    # runs: a list of runs, each the list of its rewards.
    # Gives the cumulative reward G1 of every run, in a list.
    G1s = []
    for run in runs:
        G1s.append(...)                 # 🎯 YOUR TURN: the run's G1, with your cumulative_reward
    return G1s

In [ ]:
#@title 💡 Hint 1 for task I (double-click to open) { display-mode: "form" }
# cumulative_reward(run) is the G1 of one run, a single number. Append it to G1s.

In [ ]:
#@title 💡 Hint 2 for task I (double-click to open) { display-mode: "form" }
#     G1s.append(cumulative_reward(run))

In [ ]:
# Check task I
check_cumulative_rewards(cumulative_rewards)

### 🎯 Task J. The average

Write `average(numbers)`: the total of the numbers divided by how many there are, `len(numbers)`.

In [ ]:
# 🎯 YOUR TURN, task J: the average
def average(numbers):
    # numbers: a list of numbers. Gives their average: their total divided by how many there are.
    total = 0
    for x in numbers:
        total = ...                     # 🎯 YOUR TURN: add x to the total
    return ...                          # 🎯 YOUR TURN: the total divided by how many numbers there are

In [ ]:
#@title 💡 Hint 1 for task J (double-click to open) { display-mode: "form" }
# The total grows by x in every round of the loop. After the loop, divide it by len(numbers), how many numbers
# there are.

In [ ]:
#@title 💡 Hint 2 for task J (double-click to open) { display-mode: "form" }
#     total = total + x
#     return total / len(numbers)

In [ ]:
# Check task J
check_average(average)

In [ ]:
# 1,000 runs of my_table's policy and their average G1: run it a few times
runs = play(play_runs, my_table, WEEKS, RUNS)
G1s = play(cumulative_rewards, runs)
show_money("Their G1s:", G1s)
show_money("The average G1:", play(average, G1s))

**Reading the result.** Each time the cell runs, 1,000 new runs give a new average: ten sets of 1,000 runs of
`my_table` gave averages from -\$12.20 to \$12.52. To compare two tables of bets, page 3 played both on the same runs;
section 7 does the same.

## 7. The table of bets

proto-REINFORCE does not change chances directly. It keeps a table of bets, one number for each state and
action, and softmax turns each row of bets into a row of chances: the larger a bet, the larger its chance. Page 3
started from bets picked at random between -10 and 10; `start_bets()` gives a fresh copy of them.

In [ ]:
# Page 3's table of bets
bets = start_bets()
show_bets(bets)

### Softmax

The folded cell below holds softmax, given: `softmax(row_of_bets)` turns a row of bets into a row of chances, and
`probability_table(bets)` does it for every row. Run it. Reading its code is optional, for illustration only.

In [ ]:
#@title Softmax, given: optional to read (run me) { display-mode: "form" }
# Softmax, given. Reading this cell is optional, for illustration only.

def softmax(row_of_bets):
    # A row of bets to a row of chances: e to the power of each bet, divided by their total, so that the chances
    # add up to 1 and a larger bet gets a larger chance. The largest bet is taken away first, as on page 3: the
    # chances stay the same, and exp never meets a number too large for it.
    largest = max(row_of_bets)
    powers = []
    for bet in row_of_bets:
        powers.append(exp(bet - largest))
    total = 0
    for power in powers:
        total = total + power
    chances = []
    for power in powers:
        chances.append(power / total)
    return chances


def probability_table(bets):
    # A table of bets to its probability table: softmax on every row.
    table = []
    for row in bets:
        table.append(softmax(row))
    return table


print("✅ softmax and probability_table are ready.")

In [ ]:
# The probability table of page 3's bets
show_chances(probability_table(bets))

**Reading the result.** SHAKY's bet on REPLACE, 10, is far above its other two bets, so SHAKY's chance of
REPLACE is almost 1. WORN's bets on RUN and SERVICE, -0.7 and -0.8, are close, and so are their chances.

### 🎯 Warm-up 4. A bet changed by hand

In a copy of page 3's bets, set the bet of HEALTHY and REPLACE to 5, and see what it does to HEALTHY's chances.

In [ ]:
# 🎯 YOUR TURN, warm-up 4: a bet changed by hand
my_bets = start_bets()                  # a fresh copy of page 3's bets
...                                     # 🎯 YOUR TURN: set the bet of HEALTHY and REPLACE to 5
show_bets(my_bets)
print()
show_chances(probability_table(my_bets))

In [ ]:
#@title 💡 Hint 1 for warm-up 4 (double-click to open) { display-mode: "form" }
# A bet of the table is my_bets[state][action]: the row first, then the column. Give it a new value with =.

In [ ]:
#@title 💡 Hint 2 for warm-up 4 (double-click to open) { display-mode: "form" }
#     my_bets[HEALTHY][REPLACE] = 5

In [ ]:
# Check warm-up 4
check_my_bets(my_bets)

### 🎯 Task K. The average $G_1$ of a table of bets

Write `average_G1(bets)`: the probability table of the bets, `RUNS` runs of `WEEKS` weeks with it, the $G_1$ of
every run, and their average in cents, as on page 3. Its first line is given: `same_luck()` starts the luck again
from its beginning, so every average plays the same runs, and two tables of bets are compared on the same luck.

In [ ]:
# 🎯 YOUR TURN, task K: the average G1 of a table of bets
def average_G1(bets):
    # bets: a table of bets. Gives the average G1 of its policy over RUNS runs of WEEKS weeks, in cents.
    same_luck()                         # every average plays the same luck: the same runs for every table of bets
    policy = ...                        # 🎯 YOUR TURN: the probability table of the bets
    runs = ...                          # 🎯 YOUR TURN: RUNS runs of WEEKS weeks with the policy, with your play_runs
    G1s = ...                           # 🎯 YOUR TURN: the G1 of every run, with your cumulative_rewards
    return round(average(G1s), 2)       # their average, in cents as on page 3

In [ ]:
#@title 💡 Hint 1 for task K (double-click to open) { display-mode: "form" }
# Each line calls one function you have: probability_table for the policy, play_runs for the runs,
# cumulative_rewards for their G1.
# play_runs takes the policy, the weeks of a run, then how many runs: WEEKS and RUNS.

In [ ]:
#@title 💡 Hint 2 for task K (double-click to open) { display-mode: "form" }
#     policy = probability_table(bets)
#     runs = play_runs(policy, WEEKS, RUNS)
#     G1s = cumulative_rewards(runs)

In [ ]:
# Check task K
check_average_G1(average_G1)

In [ ]:
# The average G1 of page 3's bets and of yours: run it twice
show_money("Page 3's bets:", play(average_G1, bets))
show_money("Your bets:    ", play(average_G1, my_bets))

**Reading the result.** Page 3's bets give -\$172.24, the first point of page 3's reward plot, and the same numbers
come back every time the cell runs: every average replays the same luck. Your bets replace a healthy van in
almost every week, at \$130 a time: -\$945.97.

## 8. A bet at random

Each update of proto-REINFORCE picks one bet at random, every bet as likely as any other: a row, the state, and
a column, the action. Python's `random.randint(0, 3)` gives 0, 1, 2 or 3, each as likely as the others: a row.
`random.randint(0, 2)` gives a column. Run the next cell a few times.

In [ ]:
# A row at random, then a column at random
state = random.randint(0, 3)
action = random.randint(0, 2)
print("The bet picked at random:", STATES[state], "and", ACTIONS[action])

### 🎯 Task L. A bet at random

Write `pick_bet()`: a row at random, then a column at random, given back together. It takes no inputs.

In [ ]:
# 🎯 YOUR TURN, task L: a bet at random
def pick_bet():
    # Gives a bet picked at random, every bet as likely as any other: its row, the state, and its column, the action.
    state = ...                         # 🎯 YOUR TURN: a row at random, one of the 4 states
    action = ...                        # 🎯 YOUR TURN: a column at random, one of the 3 actions
    return state, action

In [ ]:
#@title 💡 Hint 1 for task L (double-click to open) { display-mode: "form" }
# The two lines of the cell above: random.randint(0, 3) for one of the 4 rows, random.randint(0, 2) for one of
# the 3 columns.

In [ ]:
#@title 💡 Hint 2 for task L (double-click to open) { display-mode: "form" }
#     state = random.randint(0, 3)
#     action = random.randint(0, 2)

In [ ]:
# Check task L
check_pick_bet(pick_bet)

In [ ]:
# Your pick_bet, 1,200 times
count_picks(1200)

**Reading the result.** Each of the 12 bets comes about 100 times in 1,200 picks: every bet is as likely as any
other.

## 9. Before and After

Page 3's update 1 picked the bet of HEALTHY and RUN, 0.4. proto-REINFORCE asks how the average $G_1$ changes
when that bet rises a little, by a hop of 0.01. Before is the average $G_1$ of the bets as they are. After is the
average $G_1$ with that one bet a hop higher, on the same luck. The next cell works it out.

In [ ]:
# Update 1 of page 3, by hand
hop = 0.01                                      # the hop: a small change of one bet
bets = start_bets()                             # page 3's table of bets
state, action = HEALTHY, RUN                    # the bet that page 3's update 1 picked at random
hopped = copy_bets(bets)                        # a copy: the table itself stays as it is
hopped[state][action] = hopped[state][action] + hop        # one hop higher

before = play(average_G1, bets)                 # Before: the bets as they are
after = play(average_G1, hopped)                # After: one bet a hop higher, on the same luck
show_money("Before:", before)
show_money("After: ", after)

**Reading the result.** The hop raised the average $G_1$ from -\$172.24 to -\$170.50, by \$1.74. The slope, $\nabla$ (nabla), is
the change divided by the hop, rounded to a whole number as on page 3:

$$\nabla = \frac{\text{After} - \text{Before}}{\text{hop}} = \frac{1.74}{0.01} = 174$$

A positive slope says: raise this bet. A negative slope says: lower it.

### 🎯 Task M. The slope

Write `slope(bets, state, action, hop)`: Before, After with the bet a hop higher in a copy of the bets, and the
change divided by the hop.

In [ ]:
# 🎯 YOUR TURN, task M: the slope, nabla
def slope(bets, state, action, hop):
    # How fast the average G1 rises when the bet of state and action rises, measured with a hop.
    # Gives nabla, a whole number, as on page 3.
    before = average_G1(bets)                   # Before: the bets as they are
    hopped = copy_bets(bets)                    # a copy: the table itself stays as it is
    hopped[state][action] = ...                 # 🎯 YOUR TURN: the bet, one hop higher
    after = ...                                 # 🎯 YOUR TURN: After, the average G1 of the hopped bets
    nabla = ...                                 # 🎯 YOUR TURN: the change, after - before, divided by the hop
    return round(nabla)

In [ ]:
#@title 💡 Hint 1 for task M (double-click to open) { display-mode: "form" }
# The cell above, inside a function: one hop added to the bet of the copy, After from average_G1 of the copy,
# then the change divided by the hop.

In [ ]:
#@title 💡 Hint 2 for task M (double-click to open) { display-mode: "form" }
#     hopped[state][action] = hopped[state][action] + hop
#     after = average_G1(hopped)
#     nabla = (after - before) / hop

In [ ]:
# Check task M
check_slope(slope)

In [ ]:
# The slope of page 3's update 1
nabla = play(slope, bets, HEALTHY, RUN, hop)
show_value("nabla =", nabla)

**Reading the result.** nabla = 174: near this bet, the average $G_1$ rises by \$174 for each unit of bet.

## 10. The update

A positive slope says: raise this bet. The update moves the bet by $\alpha$ times the slope:

$$w = w + \alpha \nabla$$

In weekend 2 a loss had to go down: $w - \alpha \nabla$. Here the average $G_1$ has to go up:
$w + \alpha \nabla$. $\alpha = 0.01$ keeps each move small. Page 3's update 1:
$w = 0.40 + 0.01 \times 174 = 2.14$.

### 🎯 Task N. The update of one bet

Write `update_bet(bets, state, action, nabla, alpha)`: the new bet, $w + \alpha \nabla$. The given lines put it
into the table, in cents, and give it back.

In [ ]:
# 🎯 YOUR TURN, task N: the update of one bet
def update_bet(bets, state, action, nabla, alpha):
    # The update of one bet, w = w + alpha nabla, in cents. Changes the table of bets and gives the new bet.
    w = bets[state][action]                     # the bet as it is
    new_bet = ...                               # 🎯 YOUR TURN: w plus alpha times nabla
    bets[state][action] = round(new_bet, 2)     # into the table, in cents, as on page 3
    return bets[state][action]

In [ ]:
#@title 💡 Hint 1 for task N (double-click to open) { display-mode: "form" }
# w is the bet as it is, alpha the step and nabla the slope: the new bet is w plus alpha times nabla.

In [ ]:
#@title 💡 Hint 2 for task N (double-click to open) { display-mode: "form" }
#     new_bet = w + alpha * nabla

In [ ]:
# Check task N
check_update_bet(update_bet)

💭 **Think first.** With 2.14 in place of 0.40, softmax gives HEALTHY new chances. Will the average $G_1$ go up
or down?

In [ ]:
# Page 3's update 1: the new bet, the new chances, and the average G1 again, on the same luck
alpha = 0.01                                    # alpha: keeps each move small
bets = start_bets()                             # page 3's table of bets
new_bet = play(update_bet, bets, HEALTHY, RUN, nabla, alpha)
if new_bet is not None:                         # once update_bet works
    show_value("The new bet of HEALTHY and RUN:", new_bet)
    print()
    show_chances(probability_table(bets))
    print()
    show_money("The average G1 after update 1:", play(average_G1, bets))

**Reading the result.** Up: from -\$172.24 to \$73.57, the second point of page 3's reward plot. HEALTHY's chance of RUN
went from 0.47 to 0.84: one bet moved, and on average the van now makes money.

## 11. proto-REINFORCE

Now the whole loop of page 3. Each update plays new runs, picks a bet at random, measures its slope, updates the
bet, and adds one point to the reward plot: the average $G_1$ after the update.

### 🎯 Task O. The loop

Write `proto_reinforce(bets, updates, hop, alpha)`. Its first point and the first line of each update are given:
`new_luck()` deals the update new runs. Each line you write calls one of your functions.

In [ ]:
# 🎯 YOUR TURN, task O: proto-REINFORCE
def proto_reinforce(bets, updates, hop, alpha):
    # bets: the table of bets, changed by every update. updates: how many updates.
    # Gives the points of the reward plot: the average G1 before the first update, then after each update.
    points = [average_G1(bets)]                 # the first point: the bets as they are
    for update in range(updates):
        new_luck()                              # new runs for this update
        state, action = ...                     # 🎯 YOUR TURN: a bet at random, with your pick_bet
        nabla = ...                             # 🎯 YOUR TURN: the slope of that bet, with your slope
        ...                                     # 🎯 YOUR TURN: the update of that bet, with your update_bet
        points.append(...)                      # 🎯 YOUR TURN: the average G1 after the update
    return points

In [ ]:
#@title 💡 Hint 1 for task O (double-click to open) { display-mode: "form" }
# Your functions, in the order of an update: pick_bet for the bet, slope for nabla, update_bet for the update,
# average_G1 for the point.
# update_bet changes the table itself, so its line is the call alone.

In [ ]:
#@title 💡 Hint 2 for task O (double-click to open) { display-mode: "form" }
#         state, action = pick_bet()
#         nabla = slope(bets, state, action, hop)
#         update_bet(bets, state, action, nabla, alpha)
#         points.append(average_G1(bets))

In [ ]:
# Check task O
check_proto_reinforce(proto_reinforce)

### 300 updates

💭 **Think first.** Which action do you expect to be the likeliest in each state after 300 updates?

The next cell plays 300 updates from page 3's bets, each with three averages of 1,000 runs: under a minute on Colab.

In [ ]:
# 300 updates of proto-REINFORCE, from page 3's bets (under a minute on Colab)
bets = start_bets()
points = play(proto_reinforce, bets, 300, hop, alpha)
plot_rewards(points)

**Reading the result.** The first point is page 3's start, -\$172.24. Update 1 picked HEALTHY and RUN, the bet of page
3's update 1, on new runs: nabla = 224, the bet went from 0.40 to 2.64, and the average to \$123.82. After update 300 the
average is \$236.16. The row of dashes is the best strategy's average $G_1$ on the runs of update 300, \$244.03. In 167 of the
300 updates nabla was 0: the hop of 0.01 changed no run. Every update that picked SHAKY's bet on REPLACE, 10, is one
of them: REPLACE is almost certain already, and that bet never moves.

In [ ]:
# The table of bets after the updates, and its probability table
if points is not None:                          # once the loop above has run
    show_bets(bets)
    print()
    show_chances(probability_table(bets))

### The best strategy

After 300 updates the likeliest action in each state is RUN, SERVICE, REPLACE, REPLACE: the best strategy. Of the 81 strategies that take one
fixed action in each state, it has the highest average $G_1$ over a quarter, \$242.08, computed exactly from the van's
chances; on the runs of update 300 it gets \$244.03, the line of the plot. The table still takes another action now and
then: a FAILING van runs in 12 weeks of 100 and goes to the garage in 31. So the last point, \$236.16, sits a little
below the line.

## What you built

- **The roulette**: `make_roulette` and `where_it_lands` turn a row of chances and a ball into an action, and
  `pick_action` spins it for a state.
- **The forward pass**: `play_run` plays a run, `cumulative_reward` gives its $G_1$, and `average_G1` averages
  1,000 runs of the policy of a table of bets.
- **The backward pass**: `slope` raises one bet by a hop of 0.01, plays the same runs again and gives nabla.
- **The update**: `update_bet` moves the bet, $w = w + \alpha \nabla$.
- **The loop**: `proto_reinforce` repeats it, one bet picked at random by `pick_bet` each time, and gives the
  points of the reward plot.

Next, notebook 2 runs the same algorithm on another game of the arcade, Last-Minute Pricing:
[proto-REINFORCE, notebook 2: Last-Minute Pricing](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_2_last_minute_pricing_student.ipynb).